# Day 25: Token Cost Calculator
**Cost per 1,000 SmartDesk tickets before and after caching, trimming and model routing**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## How to use
Change the inputs in section 1 and run all cells. Every price here is a **placeholder**. Replace them with the approved price sheet before you quote numbers to anyone. No API key needed.

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q pandas matplotlib

# ===== Option B: VS Code / local =====
# pip install pandas matplotlib
print("Setup done")

## 1. NEW (v4): Inputs

In [ ]:
TICKETS_PER_DAY = 50_000

# Tokens per request for the SmartDesk FAQ answer
SYSTEM_TOKENS, CONTEXT_TOKENS, TICKET_TOKENS, OUTPUT_TOKENS = 800, 500, 200, 300
CACHED_SHARE = 0.5            # share of input tokens served from the prompt cache

# PLACEHOLDER prices per 1M tokens. Not real prices. Replace with the approved price sheet.
PRICES = {
    "llm_default": {"input": 1.00, "cached_input": 0.10, "output": 4.00},
    "slm_selfhosted": {"input": 0.05, "cached_input": 0.05, "output": 0.20},   # placeholder compute estimate
}

# Optimisation settings
TRIMMED_CONTEXT, OUTPUT_CAP = 250, 150
SLM_HANDLED_SHARE = 0.8       # share of tickets the SLM resolves without escalation
SLM_TOKENS_IN, SLM_TOKENS_OUT = 250, 5    # short classification prompt

## 2. NEW (v4): The cost function
Reasoning tokens are billed as output.

In [ ]:
def cost_per_request(model, inp, out, cached_share=0.0, reasoning=0):
    p = PRICES[model]
    cached = inp * cached_share
    return ((inp - cached) * p["input"] + cached * p["cached_input"] + (out + reasoning) * p["output"]) / 1_000_000

def per_1k(c):
    return round(c * 1000, 2)

## 3. NEW (v4): Before and after, step by step

In [ ]:
import pandas as pd

inp = SYSTEM_TOKENS + CONTEXT_TOKENS + TICKET_TOKENS
baseline = cost_per_request("llm_default", inp, OUTPUT_TOKENS)

cached = cost_per_request("llm_default", inp, OUTPUT_TOKENS, cached_share=CACHED_SHARE)

inp_trim = SYSTEM_TOKENS + TRIMMED_CONTEXT + TICKET_TOKENS
share_trim = min(1.0, CACHED_SHARE * inp / inp_trim)      # same cached prefix, smaller total
trimmed = cost_per_request("llm_default", inp_trim, OUTPUT_CAP, cached_share=share_trim)

slm_step = cost_per_request("slm_selfhosted", SLM_TOKENS_IN, SLM_TOKENS_OUT)
routed = slm_step + (1 - SLM_HANDLED_SHARE) * trimmed     # every ticket hits the SLM; some escalate

steps = pd.DataFrame([
    {"step": "Baseline", "change": "default LLM, no optimisation", "per_1k": per_1k(baseline)},
    {"step": "+ Caching", "change": f"{CACHED_SHARE:.0%} of input cached", "per_1k": per_1k(cached)},
    {"step": "+ Trimming", "change": f"context {CONTEXT_TOKENS}->{TRIMMED_CONTEXT}, output cap {OUTPUT_CAP}",
     "per_1k": per_1k(trimmed)},
    {"step": "+ Routing", "change": f"SLM handles {SLM_HANDLED_SHARE:.0%}", "per_1k": per_1k(routed)},
])
steps["per_day"] = (steps.per_1k * TICKETS_PER_DAY / 1000).round(2)
steps["saving_vs_baseline"] = (1 - steps.per_1k / steps.per_1k.iloc[0]).map("{:.0%}".format)
print("All values in placeholder price units.")
print(steps.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(steps.step, steps.per_1k, color=["#0E3B43", "#0FA3A3", "#0FA3A3", "#FF8A3D"])
ax.set_ylabel("Cost per 1,000 tickets\n(placeholder units)")
ax.set_title("SmartDesk FAQ answer: cost after each lever")
for i, v in enumerate(steps.per_1k):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 4. NEW (v4): Sensitivity: how much does each input matter?

In [ ]:
rows = []
for share in [0.0, 0.25, 0.5, 0.75]:
    for slm_share in [0.5, 0.7, 0.9]:
        t = cost_per_request("llm_default", inp_trim, OUTPUT_CAP, cached_share=min(1.0, share * inp / inp_trim))
        rows.append({"cached_share": share, "slm_handled": slm_share, "per_1k": per_1k(slm_step + (1 - slm_share) * t)})
grid = pd.DataFrame(rows).pivot(index="cached_share", columns="slm_handled", values="per_1k")
print("Cost per 1,000 tickets (placeholder units)")
print(grid.to_string())

## 5. NEW (v4): FinOps: budget and quota check
A monthly budget with alerts at 50%, 80% and 100%, and a per-user daily token quota for Codex. Numbers are placeholders.

In [ ]:
MONTHLY_BUDGET = 300.0               # placeholder units
DAILY_QUOTA_TOKENS = 400_000         # per learner, Codex, placeholder
spend_per_day = steps.per_day.iloc[-1]

alerts, total = [], 0.0
for day in range(1, 31):
    total += spend_per_day
    for level in [0.5, 0.8, 1.0]:
        if total >= level * MONTHLY_BUDGET and not any(a[1] == level for a in alerts):
            alerts.append((day, level))
print(f"Projected month: {total:.2f} of {MONTHLY_BUDGET:.0f} (placeholder units)")
for day, level in alerts:
    print(f"day {day:>2}: budget alert at {level:.0%}")

codex_usage = {"ana": [120_000, 350_000, 410_000], "kenji": [90_000, 80_000, 60_000]}
for user, days in codex_usage.items():
    for d, used in enumerate(days, 1):
        state = "block" if used >= DAILY_QUOTA_TOKENS else "warn" if used >= 0.8 * DAILY_QUOTA_TOKENS else "ok"
        if state != "ok":
            print(f"day {d}: {user} {state} ({used:,} of {DAILY_QUOTA_TOKENS:,} tokens)")

**TODO (cost report):** copy the before and after table into your SmartDesk cost report, add the SLM vs LLM comparison table from the other notebook, and write two sentences on the risks of each lever.

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)